# Treinamento Holdout (60/20/20):

Este notebook realiza o treinamento de holdout utilizando o dataset **SemClinBr** bruto, respeitando uma divisão de documentos de **60% treino, 20% validação e 20% teste** ordenados de forma determinística por nome do arquivo XML.


In [ ]:
!pip install -q transformers datasets peft evaluate seqeval accelerate scikit-learn

In [ ]:
import os
import sys
import json
import torch
import numpy as np
import evaluate
from datasets import Dataset, DatasetDict
from transformers import (
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
    BertTokenizerFast
)
from peft import LoraConfig, get_peft_model, TaskType

In [ ]:
esta_no_colab = 'google.colab' in sys.modules

if esta_no_colab:
    from google.colab import drive
    drive.mount('/content/drive')
    diretorio_base = '/content/drive/MyDrive/ajuste-fino'
else:
    diretorio_base = '.'

checkpoint_modelo = "medicalai/ClinicalBERT"
nome_experimento = "exp_clinicalbert_lora_60_20_20_bruto"

def obter_caminho_existente(caminho_relativo, caminho_kaggle):
    caminho_local = os.path.join(diretorio_base, caminho_relativo)
    if os.path.exists(caminho_local):
        return caminho_local
    if os.path.exists(caminho_kaggle):
        return caminho_kaggle
    return caminho_local

caminho_dados_treino = obter_caminho_existente(
    "dataset/grupos_sgr/train.bio", 
    "/kaggle/input/datasets/tonellithiago/dataset/dataset/grupos_sgr/train.bio"
)
caminho_dados_validacao = obter_caminho_existente(
    "dataset/grupos_sgr/val.bio", 
    "/kaggle/input/datasets/tonellithiago/dataset/dataset/grupos_sgr/val.bio"
)
caminho_dados_teste = obter_caminho_existente(
    "dataset/grupos_sgr/test.bio", 
    "/kaggle/input/datasets/tonellithiago/dataset/dataset/grupos_sgr/test.bio"
)

#Cria o diretório de saída sequencial
contador = 1
diretorio_saida = os.path.join(diretorio_base, f"experimentos/{nome_experimento}_{contador}")
while os.path.exists(diretorio_saida):
    contador = contador + 1
    diretorio_saida = os.path.join(diretorio_base, f"experimentos/{nome_experimento}_{contador}")

os.makedirs(diretorio_saida, exist_ok=True)
dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Utilizando hardware: {dispositivo}")
print(f"Caminho Treino: {caminho_dados_treino}")
print(f"Caminho Validação: {caminho_dados_validacao}")
print(f"Caminho Teste: {caminho_dados_teste}")

In [ ]:
#Funções auxiliares para leitura de arquivos BIO e alinhamento de tokens
def carregar_arquivo_bio(caminho_arquivo):
    lista_dados = []
    tokens_atuais = []
    labels_atuais = []
    
    if not os.path.exists(caminho_arquivo):
        print(f"Aviso: Arquivo não encontrado: {caminho_arquivo}")
        return []

    with open(caminho_arquivo, 'r', encoding='utf-8') as f:
        for linha in f:
            linha_limpa = linha.strip()
            if not linha_limpa:
                if len(tokens_atuais) > 0:
                    lista_dados.append({"tokens": tokens_atuais, "ner_tags": labels_atuais})
                    tokens_atuais = []
                    labels_atuais = []
                continue
            
            partes = linha_limpa.split()
            if len(partes) >= 2:
                tokens_atuais.append(partes[0])
                labels_atuais.append(partes[1])
                
    if len(tokens_atuais) > 0:
        lista_dados.append({"tokens": tokens_atuais, "ner_tags": labels_atuais})
        
    return lista_dados

def converter_rotulos_para_ids(exemplo):
    exemplo["ner_tags"] = [rotulo_para_id[r] for r in exemplo["ner_tags"]]
    return exemplo

def tokenizar_e_alinhar_labels(exemplos):
    entradas_tokenizadas = tokenizador(
        exemplos["tokens"], 
        truncation=True, 
        max_length=256, 
        is_split_into_words=True
    )

    lista_labels = []
    for i, labels in enumerate(exemplos["ner_tags"]):
        id_palavras = entradas_tokenizadas.word_ids(batch_index=i)
        indice_palavra_anterior = None
        id_labels_token = []
        
        for idx in id_palavras:
            if idx is None:
                id_labels_token.append(-100)
            elif idx != indice_palavra_anterior:
                id_labels_token.append(labels[idx])
            else:
                id_labels_token.append(-100)
            indice_palavra_anterior = idx
            
        lista_labels.append(id_labels_token)
        
    entradas_tokenizadas["labels"] = lista_labels
    return entradas_tokenizadas

#Funções de avaliação com seqeval
avaliador_seqeval = evaluate.load("seqeval")

def calcular_metricas(predicoes_e_labels):
    from seqeval.metrics.sequence_labeling import get_entities
    predicoes, labels_reais = predicoes_e_labels
    predicoes = np.argmax(predicoes, axis=2)

    predicoes_verdadeiras = []
    labels_verdadeiros = []
    
    for predicao, label in zip(predicoes, labels_reais):
        linha_pred = []
        linha_label = []
        for p, l in zip(predicao, label):
            if l != -100:
                linha_pred.append(lista_rotulos[p])
                linha_label.append(lista_rotulos[l])
        predicoes_verdadeiras.append(linha_pred)
        labels_verdadeiros.append(linha_label)

    tp_global = 0
    fp_global = 0
    fn_global = 0
    
    tps_por_tipo = {}
    fps_por_tipo = {}
    fns_por_tipo = {}
    
    for seq_pred, seq_label in zip(predicoes_verdadeiras, labels_verdadeiros):
        preds_ent = get_entities(seq_pred)
        labels_ent = get_entities(seq_label)
        
        preds_decomposed = set()
        for tipo, inicio, fim in preds_ent:
            for sub_tipo in tipo.split("_"):
                preds_decomposed.add((sub_tipo, inicio, fim))
                
        labels_decomposed = set()
        for tipo, inicio, fim in labels_ent:
            for sub_tipo in tipo.split("_"):
                labels_decomposed.add((sub_tipo, inicio, fim))
                
        tps = preds_decomposed.intersection(labels_decomposed)
        fps = preds_decomposed - labels_decomposed
        fns = labels_decomposed - preds_decomposed
        
        tp_global += len(tps)
        fp_global += len(fps)
        fn_global += len(fns)
        
        for tipo, _, _ in tps:
            tps_por_tipo[tipo] = tps_por_tipo.get(tipo, 0) + 1
        for tipo, _, _ in fps:
            fps_por_tipo[tipo] = fps_por_tipo.get(tipo, 0) + 1
        for tipo, _, _ in fns:
            fns_por_tipo[tipo] = fns_por_tipo.get(tipo, 0) + 1
            
    tipos_basicos = set()
    for rotulo in lista_rotulos:
        if rotulo != "O":
            for sub_tipo in rotulo[2:].split("_"):
                tipos_basicos.add(sub_tipo)
                
    dicionario_metricas = {}
    for tipo in sorted(list(tipos_basicos)):
        tp = tps_por_tipo.get(tipo, 0)
        fp = fps_por_tipo.get(tipo, 0)
        fn = fns_por_tipo.get(tipo, 0)
        
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
        number = tp + fn
        
        dicionario_metricas[f"{tipo}_precision"] = precision
        dicionario_metricas[f"{tipo}_recall"] = recall
        dicionario_metricas[f"{tipo}_f1"] = f1
        dicionario_metricas[f"{tipo}_number"] = number
        
    precision_global = tp_global / (tp_global + fp_global) if (tp_global + fp_global) > 0 else 0.0
    recall_global = tp_global / (tp_global + fn_global) if (tp_global + fn_global) > 0 else 0.0
    f1_global = 2 * (precision_global * recall_global) / (precision_global + recall_global) if (precision_global + recall_global) > 0 else 0.0
    
    total_tokens = 0
    tokens_corretos = 0
    for seq_pred, seq_label in zip(predicoes_verdadeiras, labels_verdadeiros):
        for p, l in zip(seq_pred, seq_label):
            total_tokens += 1
            if p == l:
                tokens_corretos += 1
    accuracy_global = tokens_corretos / total_tokens if total_tokens > 0 else 0.0
    
    dicionario_metricas["precision"] = precision_global
    dicionario_metricas["recall"] = recall_global
    dicionario_metricas["f1"] = f1_global
    dicionario_metricas["accuracy"] = accuracy_global
    
    return dicionario_metricas

In [ ]:
#Carrega os conjuntos de dados já divididos originalmente
dados_treino = carregar_arquivo_bio(caminho_dados_treino)
dados_validacao = carregar_arquivo_bio(caminho_dados_validacao)
dados_teste = carregar_arquivo_bio(caminho_dados_teste)

print(f"Sentenças no Treino: {len(dados_treino)}")
print(f"Sentenças na Validação: {len(dados_validacao)}")
print(f"Sentenças no Teste: {len(dados_teste)}")

#Cria o conjunto de todos os rótulos
dados_totais = dados_treino + dados_validacao + dados_teste
conjunto_rotulos = set()
for item in dados_totais:
    for rotulo in item["ner_tags"]:
        conjunto_rotulos.add(rotulo)
        
lista_rotulos = sorted(list(conjunto_rotulos))
rotulo_para_id = {}
id_para_rotulo = {}
for i, rotulo in enumerate(lista_rotulos):
    rotulo_para_id[rotulo] = i
    id_para_rotulo[i] = rotulo

print(f"Total de rótulos: {len(lista_rotulos)} ({lista_rotulos})")

dataset_treino = Dataset.from_list(dados_treino).map(converter_rotulos_para_ids)
dataset_validacao = Dataset.from_list(dados_validacao).map(converter_rotulos_para_ids)
dataset_teste = Dataset.from_list(dados_teste).map(converter_rotulos_para_ids)

datasets_combinados = DatasetDict({
    "train": dataset_treino,
    "validation": dataset_validacao,
    "test": dataset_teste
})

#Inicializa o Tokenizador
tokenizador = BertTokenizerFast.from_pretrained(checkpoint_modelo)

datasets_tokenizados = datasets_combinados.map(
    tokenizar_e_alinhar_labels,
    batched=True,
    remove_columns=datasets_combinados["train"].column_names
)

In [ ]:
#Inicializa o modelo com a arquitetura LoRA
modelo = AutoModelForTokenClassification.from_pretrained(
    checkpoint_modelo, 
    num_labels=len(lista_rotulos),
    id2label=id_para_rotulo,
    label2id=rotulo_para_id
)

configuracao_lora = LoraConfig(
    task_type=TaskType.TOKEN_CLS, 
    inference_mode=False, 
    r=16, 
    lora_alpha=16, 
    lora_dropout=0.1,
    target_modules=["q_lin", "v_lin"]
)

modelo_peft = get_peft_model(modelo, configuracao_lora)
modelo_peft.print_trainable_parameters()

#Importa o callback para parada antecipada
from transformers import EarlyStoppingCallback

#Argumentos de Treinamento
argumentos_treinamento = TrainingArguments(
    output_dir=os.path.join(diretorio_saida, "checkpoints"),
    eval_strategy="epoch",
    learning_rate=3e-5,               # taxa de aprendizado
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=10,            
    weight_decay=0.01,
    save_strategy="epoch",
    load_best_model_at_end=True,     #Necessário para Early Stopping
    metric_for_best_model="f1",      #Seleciona pelo F1 de entidade
    greater_is_better=True,          #Maior F1 é melhor
    push_to_hub=False,
    warmup_ratio=0.1,
    report_to="none"
)

#Inicializa o Treinador do HuggingFace com EarlyStoppingCallback
treinador = Trainer(
    model=modelo_peft,
    args=argumentos_treinamento,
    train_dataset=datasets_tokenizados["train"],
    eval_dataset=datasets_tokenizados["validation"],
    data_collator=DataCollatorForTokenClassification(tokenizador),
    compute_metrics=calcular_metricas,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)] 
)

print("Iniciando treinamento...")
treinador.train()

print("Avaliando no conjunto de teste...")
resultados_teste = treinador.evaluate(datasets_tokenizados["test"])
print(resultados_teste)

In [ ]:
#Imprime as métricas por categoria no console
print("\nMétricas por Categoria no Teste:")
chaves_metricas = resultados_teste.keys()
for chave in sorted(chaves_metricas):
    if chave.startswith("eval_") and chave.endswith("_f1") and chave != "eval_f1":
        cat = chave[5:-3]
        print(f"  {cat}: F1={resultados_teste[f'eval_{cat}_f1']:.4f} | Precision={resultados_teste[f'eval_{cat}_precision']:.4f} | Recall={resultados_teste[f'eval_{cat}_recall']:.4f}")

#Salva os resultados finais em JSON
caminho_resultados = os.path.join(diretorio_saida, "test_results.json")
with open(caminho_resultados, "w") as f:
    json.dump(resultados_teste, f, indent=4)

#Salva relatório legível em TXT
caminho_relatorio = os.path.join(diretorio_saida, "test_report.txt")
with open(caminho_relatorio, "w", encoding="utf-8") as f:
    f.write(f"RELATÓRIO FINAL TESTE (Divisão 60/20/20 por Documentos) - {nome_experimento}\n")
    f.write(f"{'='*50}\n")
    f.write(f"Modelo: {checkpoint_modelo}\n")
    f.write(f"Parâmetros do LoRA:\n")
    f.write(f"  r: {configuracao_lora.r}\n")
    f.write(f"  lora_alpha: {configuracao_lora.lora_alpha}\n")
    f.write(f"  lora_dropout: {configuracao_lora.lora_dropout}\n")
    f.write(f"  target_modules: {configuracao_lora.target_modules}\n")
    f.write(f"Parâmetros de Treinamento:\n")
    f.write(f"  Learning Rate: {argumentos_treinamento.learning_rate}\n")
    f.write(f"  Batch Size Treino: {argumentos_treinamento.per_device_train_batch_size}\n")
    f.write(f"  Batch Size Eval: {argumentos_treinamento.per_device_eval_batch_size}\n")
    f.write(f"  Épocas: {argumentos_treinamento.num_train_epochs}\n")
    f.write(f"  Weight Decay: {argumentos_treinamento.weight_decay}\n")
    f.write(f"  Semente: {argumentos_treinamento.seed}\n")
    f.write(f"{'='*50}\n")
    f.write(f"F1 Geral: {resultados_teste['eval_f1']:.4f}\n")
    f.write(f"Precision Geral: {resultados_teste['eval_precision']:.4f}\n")
    f.write(f"Recall Geral: {resultados_teste['eval_recall']:.4f}\n")
    f.write(f"Accuracy Geral: {resultados_teste['eval_accuracy']:.4f}\n\n")
    f.write(f"Métricas por Categoria:\n")
    for chave in sorted(chaves_metricas):
        if chave.startswith("eval_") and chave.endswith("_f1") and chave != "eval_f1":
            cat = chave[5:-3]
            f.write(f"  {cat}:\n")
            f.write(f"    F1: {resultados_teste[f'eval_{cat}_f1']:.4f}\n")
            f.write(f"    Precision: {resultados_teste[f'eval_{cat}_precision']:.4f}\n")
            f.write(f"    Recall: {resultados_teste[f'eval_{cat}_recall']:.4f}\n")

modelo_peft.save_pretrained(os.path.join(diretorio_saida, "model_final"))
tokenizador.save_pretrained(os.path.join(diretorio_saida, "model_final"))

print(f"Treinamento concluído. Resultados salvos em {diretorio_saida}")